In [7]:
# Imports
from torchvision import datasets
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from joblib import dump
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import os
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
from joblib import load


# Train / Test Arrays to append into for MNIST
XTrain = []
yTrain = []
XTest = []
yTest = []

# Normalize Transform
normalize_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.5,), std=(0.5,))
])

# Train the model off MNIST
trainDataset = datasets.MNIST("./MNIST", train=True,
transform=normalize_transform, download=True)
testDataset = datasets.MNIST("./MNIST", train=False,
transform=normalize_transform, download=True)

# Train Dataset
for img, label in trainDataset:
    XTrain.append(img.numpy().flatten())
    yTrain.append(label)

# Test Dataset
for img, label in testDataset:
    XTest.append(img.numpy().flatten())
    yTest.append(label)



start
normalize transform starting


BEGIN...load data and transform

In [8]:
# Scaled (testing to see if it runs better w/ this or if I just need the normalize in the start)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(XTrain)
X_test_scaled = scaler.transform(XTest)


In [ ]:
# Actual SVM model, play around with random state, etc. from GeeksForGeeks.
svm_classifier = SVC(kernel='linear', C=1.0, random_state=42)
svm_classifier.fit(X_train_scaled, yTrain)

# Save the model with joblib.
dump({
    "model": svm_classifier,
    "scaler": scaler
}, "svm_mnist.joblib")

In [ ]:
# Accuracy test with table.
yPred = svm_classifier.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(yTest, yPred):.2f}")
print(classification_report(yTest, yPred))

In [ ]:
# Everything to do with the filename, etc is from Nate Sharp as I could not figure out how to do it. I may change it up a little bit later but for now it's his code that he gave me to use.
imageDirectory = "./IMAGES"
correct = 0
total = 0

data = load("svm_mnist.joblib")

svm_classifier = data["model"]
scaler = data["scaler"]

X_new_scaled = scaler.fit_transform(XTest)

for filename in os.listdir(imageDirectory):
    labels = int(filename[0])
    path = os.path.join(imageDirectory, filename)
    img = Image.open(path).convert("L")
    arr = np.array(img).flatten() / 255.0
    sample = arr.reshape(1, -1)
    svmPrediction = svm_classifier.predict(sample)[0]
    if svmPrediction == labels:
        correct += 1
    total += 1

print(f"Total: {total}")
print(f"Correct: {correct}")
